# Comparaison SeapoPym DAG vs Seapodym-LMTL (Pacifique)

Ce notebook analyse les résultats des simulations Pacifique pour valider le module de transport couplé.

**Objectifs** :

1.  **Validation** : Comparer la simulation DAG (avec transport) avec la référence Seapodym-LMTL.
2.  **Impact du Transport** : Comparer la simulation DAG avec transport vs sans transport pour quantifier l'effet physique.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import pint_xarray

# === CONFIGURATION DES CHEMINS ===
BASE_DIR = Path("/Users/adm-lehodey/Documents/Workspace/Projects/seapopym-message/data/article")
DATA_DIR = BASE_DIR / "data"
FIGURES_DIR = BASE_DIR / "figures"
SUMMARY_DIR = BASE_DIR / "summary"
FIGURES_DIR.mkdir(exist_ok=True)

# Fichiers de données
FILE_REF = DATA_DIR / "seapodym_lmtl_output_pacific_ref.zarr"
FILE_TRANS = DATA_DIR / "seapopym_pacific_transport.zarr"
FILE_NO_TRANS = DATA_DIR / "seapopym_pacific_no_transport.zarr"

# Sortie résumée
SUMMARY_FILE = SUMMARY_DIR / "notebook_05e_comparison_summary.txt"

print(f"Ref: {FILE_REF}")
print(f"Transport: {FILE_TRANS}")
print(f"No-Transport: {FILE_NO_TRANS}")

## 1. Chargement des Données


In [ ]:
# Chargement Reference (Zooplankton)
ds_ref = xr.open_zarr(FILE_REF)
if "zooplankton" in ds_ref:
    ref = ds_ref["zooplankton"].load()
else:
    raise ValueError("Variable 'zooplankton' not found in reference file.")

# Chargement DAG Transport
ds_trans = xr.open_zarr(FILE_TRANS)
dag_trans = ds_trans["biomass"].load()

# Chargement DAG No-Transport
ds_no_trans = xr.open_zarr(FILE_NO_TRANS)
dag_no_trans = ds_no_trans["biomass"].load()

# Standardisation des noms de coordonnées
# Les exports utilisent "time", "y", "x"
# La référence utilise "time", "latitude", "longitude"
# On harmonise tout vers "time", "latitude", "longitude"
rename_dict = {"y": "latitude", "x": "longitude"}

ref = ref.rename({k: v for k, v in rename_dict.items() if k in ref.dims})
dag_trans = dag_trans.rename({k: v for k, v in rename_dict.items() if k in dag_trans.dims})
dag_no_trans = dag_no_trans.rename({k: v for k, v in rename_dict.items() if k in dag_no_trans.dims})

print("Données chargées.")
print(f"Ref shape: {ref.shape}")
print(f"Trans shape: {dag_trans.shape}")
print(f"No-Trans shape: {dag_no_trans.shape}")
print(f"Ref coords: {list(ref.dims)}")
print(f"Trans coords: {list(dag_trans.dims)}")
print(f"No-Trans coords: {list(dag_no_trans.dims)}")

In [ ]:
# dag_no_trans = dag_no_trans.sel(latitude=slice(-40, 55))
# dag_trans = dag_trans.sel(latitude=slice(-40, 55))
# ref = ref.sel(latitude=slice(-40, 55))


## 2. Alignement et Prétraitement

On exclut la période de spin-up (2 premières années) pour la comparaison.


In [ ]:
# Période de comparaison : 2002 - 2004 (Spin-up 2000-2001)
# Note: Les simulations couvrent 2000-01-01 à 2004-12-31
TIME_START = "2002-01-01"
TIME_END = "2004-12-31"

# Sélection et Alignement
ref_cut = ref.sel(time=slice(TIME_START, TIME_END))
dag_trans_cut = dag_trans.sel(time=slice(TIME_START, TIME_END))
dag_no_trans_cut = dag_no_trans.sel(time=slice(TIME_START, TIME_END))

# IMPORTANT: Les données SeapoPym sont en pas de temps 3-horaires
# On doit les resampler à la journée pour les aligner avec la référence
print(f"Avant resampling:")
print(f"  Référence: {len(ref_cut.time)} pas de temps")
print(f"  Transport: {len(dag_trans_cut.time)} pas de temps")
print(f"  No-Transport: {len(dag_no_trans_cut.time)} pas de temps")

# Resampling des données SeapoPym à la journée (moyenne journalière)
dag_trans_cut = dag_trans_cut.resample(time="1D").mean()
dag_no_trans_cut = dag_no_trans_cut.resample(time="1D").mean()

print(f"\nAprès resampling à la journée:")
print(f"  Transport: {len(dag_trans_cut.time)} pas de temps")
print(f"  No-Transport: {len(dag_no_trans_cut.time)} pas de temps")

# IMPORTANT: Normalisation des timestamps à 00:00 pour tous les datasets
# La référence a des timestamps à 12:00, on doit les normaliser à 00:00
print(f"\nNormalisation des timestamps à 00:00...")
ref_cut["time"] = ref_cut.time.dt.floor("D")
dag_trans_cut["time"] = dag_trans_cut.time.dt.floor("D")
dag_no_trans_cut["time"] = dag_no_trans_cut.time.dt.floor("D")

# Alignement des grilles (intersection)
ref_cut, dag_trans_cut = xr.align(ref_cut, dag_trans_cut, join="inner")
_, dag_no_trans_cut = xr.align(ref_cut, dag_no_trans_cut, join="inner")

print(f"\nAprès alignement:")
print(f"  Période alignée : {len(ref_cut.time)} jours")
print(f"  Grille alignée : {len(ref_cut.latitude)} x {len(ref_cut.longitude)}")

## 3. Métriques de Validation


In [ ]:
def compute_metrics(da_ref, da_model):
    """Calcule RMSE, NRMSE (normalisé par std) et MAPE entre référence et modèle."""
    diff = da_model - da_ref

    # RMSE
    rmse = np.sqrt((diff**2).mean()).item()

    # NRMSE (normalisé par l'écart-type de la référence)
    std_ref = da_ref.std().item()
    nrmse = rmse / std_ref

    # MAPE
    with np.errstate(divide="ignore", invalid="ignore"):
        mape = np.abs(diff / da_ref).where(da_ref > 1e-6).mean().item() * 100

    return {"RMSE": rmse, "NRMSE": nrmse, "MAPE": mape}


# === Comparaison 1 : DAG Transport vs Ref ===
metrics_trans = compute_metrics(ref_cut, dag_trans_cut)

print("=" * 60)
print("MÉTRIQUES : DAG Transport vs Référence")
print("=" * 60)
print(f"  {'RMSE':<12s} : {metrics_trans['RMSE']:6.2f}")
print(f"  {'NRMSE':<12s} : {metrics_trans['NRMSE']:6.2f}")
print(f"  {'MAPE':<12s} : {metrics_trans['MAPE']:6.2f} %")

# === Comparaison 2 : DAG No-Transport vs Ref ===
metrics_no_trans = compute_metrics(ref_cut, dag_no_trans_cut)

print("\n" + "=" * 60)
print("MÉTRIQUES : DAG No-Transport vs Référence")
print("=" * 60)
print(f"  {'RMSE':<12s} : {metrics_no_trans['RMSE']:6.2f}")
print(f"  {'NRMSE':<12s} : {metrics_no_trans['NRMSE']:6.2f}")
print(f"  {'MAPE':<12s} : {metrics_no_trans['MAPE']:6.2f} %")
print("=" * 60)

## 3.1 Cartes Spatiales de RMSE, NRMSE et MAPE

Visualisation spatiale des trois métriques d'erreur pour les deux configurations.


In [ ]:
# Calcul des RMSE, NRMSE et MAPE spatiales (moyennées sur le temps)

vmax_rmse = float(ref_cut.quantile(0.75))
vmax_nrmse = 2
vmax_mape = 100

# === DAG Transport vs Ref ===
diff_trans = dag_trans_cut - ref_cut
rmse_trans = np.sqrt((diff_trans**2).mean(dim="time"))

# NRMSE : normalisé par l'écart-type spatial de la référence
std_ref_spatial = ref_cut.std(dim="time")
nrmse_trans = rmse_trans / std_ref_spatial

# MAPE : pourcentage d'erreur absolue moyenne
with np.errstate(divide="ignore", invalid="ignore"):
    mape_trans = (np.abs(diff_trans / ref_cut).where(ref_cut > 1e-6).mean(dim="time")) * 100

# === DAG No-Transport vs Ref ===
diff_no_trans = dag_no_trans_cut - ref_cut
rmse_no_trans = np.sqrt((diff_no_trans**2).mean(dim="time"))
nrmse_no_trans = rmse_no_trans / std_ref_spatial

with np.errstate(divide="ignore", invalid="ignore"):
    mape_no_trans = (np.abs(diff_no_trans / ref_cut).where(ref_cut > 1e-6).mean(dim="time")) * 100

# === FIGURE 1 : RMSE (1x2 subplots) ===
fig1, axes1 = plt.subplots(1, 2, figsize=(16, 6))

# RMSE Transport
ax1 = axes1[0]
im1 = rmse_trans.plot(ax=ax1, cmap="viridis_r", vmin=0, vmax=vmax_rmse, add_colorbar=False)
ax1.set_title("RMSE - DAG Transport vs Référence", fontsize=12, fontweight="bold")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
cbar1 = plt.colorbar(im1, ax=ax1, orientation="vertical", pad=0.02)
cbar1.set_label("RMSE (g/m²)", fontsize=10)

# RMSE No-Transport
ax2 = axes1[1]
im2 = rmse_no_trans.plot(ax=ax2, cmap="viridis_r", vmin=0, vmax=vmax_rmse, add_colorbar=False)
ax2.set_title("RMSE - DAG No-Transport vs Référence", fontsize=12, fontweight="bold")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
cbar2 = plt.colorbar(im2, ax=ax2, orientation="vertical", pad=0.02)
cbar2.set_label("RMSE (g/m²)", fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig_05e_spatial_rmse.png", dpi=150, bbox_inches="tight")
plt.show()

# === FIGURE 2 : NRMSE (1x2 subplots) ===
fig2, axes2 = plt.subplots(1, 2, figsize=(16, 6))

# NRMSE Transport
ax3 = axes2[0]
im3 = nrmse_trans.plot(ax=ax3, cmap="viridis_r", vmin=0, vmax=vmax_nrmse, add_colorbar=False)
ax3.set_title("NRMSE - DAG Transport vs Référence", fontsize=12, fontweight="bold")
ax3.set_xlabel("Longitude")
ax3.set_ylabel("Latitude")
cbar3 = plt.colorbar(im3, ax=ax3, orientation="vertical", pad=0.02)
cbar3.set_label("NRMSE", fontsize=10)

# NRMSE No-Transport
ax4 = axes2[1]
im4 = nrmse_no_trans.plot(ax=ax4, cmap="viridis_r", vmin=0, vmax=vmax_nrmse, add_colorbar=False)
ax4.set_title("NRMSE - DAG No-Transport vs Référence", fontsize=12, fontweight="bold")
ax4.set_xlabel("Longitude")
ax4.set_ylabel("Latitude")
cbar4 = plt.colorbar(im4, ax=ax4, orientation="vertical", pad=0.02)
cbar4.set_label("NRMSE", fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig_05e_spatial_nrmse.png", dpi=150, bbox_inches="tight")
plt.show()

# === FIGURE 3 : MAPE (1x2 subplots) ===
fig3, axes3 = plt.subplots(1, 2, figsize=(16, 6))

# MAPE Transport
ax5 = axes3[0]
im5 = mape_trans.plot(ax=ax5, cmap="viridis_r", vmin=0, vmax=vmax_mape, add_colorbar=False)
ax5.set_title("MAPE - DAG Transport vs Référence", fontsize=12, fontweight="bold")
ax5.set_xlabel("Longitude")
ax5.set_ylabel("Latitude")
cbar5 = plt.colorbar(im5, ax=ax5, orientation="vertical", pad=0.02)
cbar5.set_label("MAPE (%)", fontsize=10)

# MAPE No-Transport
ax6 = axes3[1]
im6 = mape_no_trans.plot(ax=ax6, cmap="viridis_r", vmin=0, vmax=vmax_mape, add_colorbar=False)
ax6.set_title("MAPE - DAG No-Transport vs Référence", fontsize=12, fontweight="bold")
ax6.set_xlabel("Longitude")
ax6.set_ylabel("Latitude")
cbar6 = plt.colorbar(im6, ax=ax6, orientation="vertical", pad=0.02)
cbar6.set_label("MAPE (%)", fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig_05e_spatial_mape.png", dpi=150, bbox_inches="tight")
plt.show()

# Statistiques
print("=" * 60)
print("STATISTIQUES SPATIALES")
print("=" * 60)
print("\nDAG Transport vs Référence:")
print(
    f"  RMSE  - Min: {rmse_trans.min().item():.2f}, Max: {rmse_trans.max().item():.2f}, Mean: {rmse_trans.mean().item():.2f} g/m²"
)
print(
    f"  NRMSE - Min: {nrmse_trans.min().item():.2f}, Max: {nrmse_trans.max().item():.2f}, Mean: {nrmse_trans.mean().item():.2f}"
)
print(
    f"  MAPE  - Min: {mape_trans.min().item():.2f}, Max: {mape_trans.max().item():.2f}, Mean: {mape_trans.mean().item():.2f} %"
)

print("\nDAG No-Transport vs Référence:")
print(
    f"  RMSE  - Min: {rmse_no_trans.min().item():.2f}, Max: {rmse_no_trans.max().item():.2f}, Mean: {rmse_no_trans.mean().item():.2f} g/m²"
)
print(
    f"  NRMSE - Min: {nrmse_no_trans.min().item():.2f}, Max: {nrmse_no_trans.max().item():.2f}, Mean: {nrmse_no_trans.mean().item():.2f}"
)
print(
    f"  MAPE  - Min: {mape_no_trans.min().item():.2f}, Max: {mape_no_trans.max().item():.2f}, Mean: {mape_no_trans.mean().item():.2f} %"
)
print("=" * 60)

## 4. Séries Temporelles par Zone Latitudinale

On décompose l'analyse en 3 zones latitudinales :

-   **Zone Sud** : latitude < -20°
-   **Zone Tropicale** : -20° ≤ latitude ≤ 20°
-   **Zone Nord** : latitude > 20°


In [ ]:
# Définition des zones latitudinales
zones = {
    "Sud (< -20°)": (None, -20),
    "Tropicale (-20° à 20°)": (-20, 20),
    "Nord (> 20°)": (20, None),
}

# Création de la figure avec 3 sous-graphiques
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

for ax, (zone_name, (lat_min, lat_max)) in zip(axes, zones.items()):
    # Sélection de la zone latitudinale
    if lat_min is None:
        # Zone sud : lat < -20
        ref_zone = ref_cut.where(ref_cut.latitude < lat_max, drop=True)
        trans_zone = dag_trans_cut.where(dag_trans_cut.latitude < lat_max, drop=True)
        no_trans_zone = dag_no_trans_cut.where(dag_no_trans_cut.latitude < lat_max, drop=True)
    elif lat_max is None:
        # Zone nord : lat > 20
        ref_zone = ref_cut.where(ref_cut.latitude > lat_min, drop=True)
        trans_zone = dag_trans_cut.where(dag_trans_cut.latitude > lat_min, drop=True)
        no_trans_zone = dag_no_trans_cut.where(dag_no_trans_cut.latitude > lat_min, drop=True)
    else:
        # Zone tropicale : -20 <= lat <= 20
        ref_zone = ref_cut.sel(latitude=slice(lat_min, lat_max))
        trans_zone = dag_trans_cut.sel(latitude=slice(lat_min, lat_max))
        no_trans_zone = dag_no_trans_cut.sel(latitude=slice(lat_min, lat_max))

    # Calcul des séries temporelles (somme sur latitude et longitude)
    ts_ref = ref_zone.sum(["latitude", "longitude"])
    ts_trans = trans_zone.sum(["latitude", "longitude"])
    ts_no_trans = no_trans_zone.sum(["latitude", "longitude"])

    # Tracé
    ts_ref.plot(ax=ax, label="Seapodym (Ref)", linewidth=2, color="black")
    ts_trans.plot(ax=ax, label="DAG (Transport)", linewidth=1.5, linestyle="--", color="blue")
    ts_no_trans.plot(
        ax=ax, label="DAG (No Transport)", linewidth=1, linestyle=":", color="grey", alpha=0.7
    )

    ax.set_title(f"Biomasse Totale - Zone {zone_name}")
    ax.set_xlabel("Temps")
    ax.set_ylabel("Biomasse (g/m²)")
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig_05e_pacific_timeseries_zones.png", dpi=150)
plt.show()

## 5. Export des Résultats


In [ ]:
# --- Export des Résultats (Résumé) ---

summary_content = f"""
================================================================================
RÉSUMÉ DE VALIDATION : SeapoPym vs Seapodym-LMTL (Pacifique)
================================================================================
Date: {pd.Timestamp.now()}
Période: {TIME_START} à {TIME_END}
Zone latitude: [-40, 55]

MÉTRIQUES (DAG Transport vs Référence):
--------------------------------------------------------------------------------
RMSE  : {metrics_trans["RMSE"]:.2f}
NRMSE : {metrics_trans["NRMSE"]:.2f}
MAPE  : {metrics_trans["MAPE"]:.2f} %

MÉTRIQUES (DAG No-Transport vs Référence):
--------------------------------------------------------------------------------
RMSE  : {metrics_no_trans["RMSE"]:.2f}
NRMSE : {metrics_no_trans["NRMSE"]:.2f}
MAPE  : {metrics_no_trans["MAPE"]:.2f} %

FIGURES GÉNÉRÉES :
--------------------------------------------------------------------------------
- fig_05e_spatial_rmse.png : Cartes spatiales de RMSE
  * Transport vs Référence (gauche)
  * No-Transport vs Référence (droite)

- fig_05e_spatial_nrmse.png : Cartes spatiales de NRMSE
  * Transport vs Référence (gauche)
  * No-Transport vs Référence (droite)

- fig_05e_spatial_mape.png : Cartes spatiales de MAPE
  * Transport vs Référence (gauche)
  * No-Transport vs Référence (droite)

- fig_05e_pacific_timeseries_zones.png : Séries temporelles par zone latitudinale
  * Zone Sud (< -20°)
  * Zone Tropicale (-20° à 20°)
  * Zone Nord (> 20°)

================================================================================
"""

with open(SUMMARY_FILE, "w") as f:
    f.write(summary_content)

print(f"Résumé sauvegardé dans : {SUMMARY_FILE}")
print(summary_content)

In [ ]:
ds_trans["biomass"].sel(
    # time=slice("2015", "2017"),
    x=slice(229, 231),
    y=slice(48, 50),
).mean(["x", "y"]).plot(label="SeapoPym")

ds_no_trans["biomass"].sel(
    # time=slice("2015", "2017"),
    x=slice(229, 231),
    y=slice(48, 50),
).mean(["x", "y"]).plot(label="No transport")

plt.xlabel("Time")
plt.ylabel("Zooplancton biomass")
plt.legend()